# Banking on Behaviour - Starter Notebook

This notebook shows you how to load the data and create a valid submission.

Feature engineering, model selection, and validation strategy are up to you.

**Target:** Predict `next_3m_txn_count` for each customer in Test.csv

**Metric:** RMSLE (Root Mean Squared Logarithmic Error)

In [1]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Core files
train = pd.read_csv('../data/raw/Train.csv')
test  = pd.read_csv('../data/raw/Test.csv')

print(f'Train: {train.shape}, Test: {test.shape}')
train.head()

Train: (8360, 2), Test: (3584, 1)


,UniqueID,next_3m_txn_count
0,00093e2d-9e1e-4061-ad27-a79b8ff9e165,129
1,0011d60f-a4e2-4333-81fc-2d557a82109b,16
2,0016f1e2-64c1-4c65-a668-1dc6bf3b5875,117
3,001aa3c5-632d-435e-a421-cc3615ccef4d,70
4,00298c6f-4f9d-4f28-b72c-ad0e56e9eb84,393


In [3]:
# Load transactions
txn = pd.read_parquet('../data/raw/transactions_features.parquet')
txn['TransactionDate'] = pd.to_datetime(txn['TransactionDate'])
max_date = txn['TransactionDate'].max()
print(f'Transactions: {len(txn):,} rows | Date range: {txn["TransactionDate"].min()} to {max_date}')
print(txn.columns.tolist())

Transactions: 18,017,073 rows | Date range: 2012-12-25 00:00:00 to 2015-10-31 00:00:00
['UniqueID', 'AccountID', 'TransactionDate', 'TransactionAmount', 'TransactionTypeDescription', 'TransactionBatchDescription', 'StatementBalance', 'IsDebitCredit', 'ReversalTypeDescription']


In [4]:
# ============================================================
# FEATURE BLOCK 1: Overall transaction stats (all time)
# ============================================================
feat_all = txn.groupby('UniqueID').agg(
    txn_count_all   = ('TransactionAmount', 'count'),
    txn_sum_all     = ('TransactionAmount', 'sum'),
    txn_mean_all    = ('TransactionAmount', 'mean'),
    txn_std_all     = ('TransactionAmount', 'std'),
    txn_median_all  = ('TransactionAmount', 'median'),
).reset_index()

# ============================================================
# FEATURE BLOCK 2: Debit vs Credit split
# ============================================================
# The competition notes: TransactionAmount is signed — negatives = debits, positives = credits
debits  = txn[txn['TransactionAmount'] < 0]
credits = txn[txn['TransactionAmount'] > 0]

feat_debit = debits.groupby('UniqueID').agg(
    debit_count = ('TransactionAmount', 'count'),
    debit_sum   = ('TransactionAmount', 'sum'),
    debit_mean  = ('TransactionAmount', 'mean'),
).reset_index()

feat_credit = credits.groupby('UniqueID').agg(
    credit_count = ('TransactionAmount', 'count'),
    credit_sum   = ('TransactionAmount', 'sum'),
    credit_mean  = ('TransactionAmount', 'mean'),
).reset_index()

print('Debit/credit features built')

Debit/credit features built


In [ ]:
# ============================================================
# FEATURE BLOCK 3: Recency windows (6m, 3m, 1m)
# ============================================================
def window_features(df, days, suffix):
    cutoff = max_date - pd.Timedelta(days=days)
    sub = df[df['TransactionDate'] >= cutoff]
    feat = sub.groupby('UniqueID').agg(
        count = ('TransactionAmount', 'count'),
        sum   = ('TransactionAmount', 'sum'),
        mean  = ('TransactionAmount', 'mean'),
    ).reset_index()
    feat.columns = ['UniqueID', f'txn_count_{suffix}', f'txn_sum_{suffix}', f'txn_mean_{suffix}']
    return feat
feat_12m = window_features(txn, 365, '12m')
feat_6m = window_features(txn, 180, '6m')
feat_3m = window_features(txn, 90,  '3m')
feat_1m = window_features(txn, 30,  '1m')

print('Recency window features built')

Recency window features built


In [ ]:
# ============================================================
# BLOCK 4: Month-by-month counts for last 6 months
# This gives the model the SHAPE of activity, not just totals
# ============================================================
txn['year_month'] = txn['TransactionDate'].dt.to_period('M')
monthly_counts = txn.groupby(['UniqueID', 'year_month']).size().reset_index(name='monthly_count')

# Pivot: each of last 6 months becomes a column
months_sorted = sorted(txn['year_month'].unique())[-6:]
month_labels  = {m: f'month_m{i+1}' for i, m in enumerate(months_sorted)}  # m1=oldest, m6=most recent

monthly_pivot = monthly_counts[monthly_counts['year_month'].isin(months_sorted)].copy()
monthly_pivot['month_label'] = monthly_pivot['year_month'].map(month_labels)
monthly_pivot = monthly_pivot.pivot(index='UniqueID', columns='month_label', values='monthly_count').reset_index().fillna(0)
monthly_pivot.columns.name = None

# Consistency stats across ALL months
all_monthly = txn.groupby(['UniqueID', 'year_month']).size().reset_index(name='mc')
consistency = all_monthly.groupby('UniqueID')['mc'].agg(
    monthly_mean   = 'mean',
    monthly_std    = 'std',
    monthly_max    = 'max',
    monthly_min    = 'min',
    monthly_median = 'median',
    active_months  = 'count',
).reset_index()
# Coefficient of variation — lower = more consistent customer
consistency['monthly_cv'] = consistency['monthly_std'] / (consistency['monthly_mean'] + 1)

print(f'Monthly pivot shape: {monthly_pivot.shape}')
print(f'Consistency shape: {consistency.shape}')

In [ ]:
# ============================================================
# BLOCK 5: Holiday season + trend + recency
# ============================================================
# Holiday: Nov/Dec/Jan (matches prediction window)
holiday_mask = txn['TransactionDate'].dt.month.isin([11, 12, 1])
feat_holiday = txn[holiday_mask].groupby('UniqueID').agg(
    holiday_txn_count = ('TransactionAmount', 'count'),
    holiday_txn_sum   = ('TransactionAmount', 'sum'),
    holiday_txn_mean  = ('TransactionAmount', 'mean'),
).reset_index()

# Trend: last 3m vs prior 3m
cutoff_3m = max_date - pd.Timedelta(days=90)
cutoff_6m = max_date - pd.Timedelta(days=180)
recent = txn[txn['TransactionDate'] >= cutoff_3m].groupby('UniqueID')['TransactionAmount'].count().rename('cnt_recent')
prior  = txn[(txn['TransactionDate'] >= cutoff_6m) & (txn['TransactionDate'] < cutoff_3m)].groupby('UniqueID')['TransactionAmount'].count().rename('cnt_prior')
trend_df = pd.concat([recent, prior], axis=1).reset_index().fillna(0)
trend_df['txn_trend_ratio'] = (trend_df['cnt_recent'] + 1) / (trend_df['cnt_prior'] + 1)
trend_df['txn_trend_diff']  = trend_df['cnt_recent'] - trend_df['cnt_prior']
trend_df = trend_df[['UniqueID', 'txn_trend_ratio', 'txn_trend_diff']]

# Also: 12m vs prior 12m trend
cutoff_12m = max_date - pd.Timedelta(days=365)
cutoff_24m = max_date - pd.Timedelta(days=730)
recent_12 = txn[txn['TransactionDate'] >= cutoff_12m].groupby('UniqueID')['TransactionAmount'].count().rename('cnt_12m')
prior_12  = txn[(txn['TransactionDate'] >= cutoff_24m) & (txn['TransactionDate'] < cutoff_12m)].groupby('UniqueID')['TransactionAmount'].count().rename('cnt_prev_12m')
trend_12m = pd.concat([recent_12, prior_12], axis=1).reset_index().fillna(0)
trend_12m['txn_trend_ratio_12m'] = (trend_12m['cnt_12m'] + 1) / (trend_12m['cnt_prev_12m'] + 1)
trend_12m = trend_12m[['UniqueID', 'txn_trend_ratio_12m']]

# Recency + tenure
last_txn  = txn.groupby('UniqueID')['TransactionDate'].max().reset_index()
last_txn['days_since_last_txn'] = (max_date - last_txn['TransactionDate']).dt.days
last_txn  = last_txn[['UniqueID', 'days_since_last_txn']]

first_txn = txn.groupby('UniqueID')['TransactionDate'].min().reset_index()
first_txn['customer_tenure_days'] = (max_date - first_txn['TransactionDate']).dt.days
first_txn = first_txn[['UniqueID', 'customer_tenure_days']]

# avg transactions per active month
avg_per_month = feat_all[['UniqueID', 'txn_count_all']].merge(consistency[['UniqueID', 'active_months']], on='UniqueID')
avg_per_month['avg_txn_per_month'] = avg_per_month['txn_count_all'] / avg_per_month['active_months']
avg_per_month = avg_per_month[['UniqueID', 'avg_txn_per_month']]

print('Block 5 done')

Holiday season features built


In [ ]:
# ============================================================
# BLOCK 6: Transaction type features (if column exists)
# ============================================================
if 'TransactionType' in txn.columns:
    type_div = txn.groupby('UniqueID')['TransactionType'].nunique().reset_index()
    type_div.columns = ['UniqueID', 'txn_type_diversity']
    
    # Top transaction types as count features
    top_types = txn['TransactionType'].value_counts().nlargest(5).index
    type_counts = txn[txn['TransactionType'].isin(top_types)].groupby(
        ['UniqueID', 'TransactionType']
    ).size().unstack(fill_value=0).reset_index()
    type_counts.columns = ['UniqueID'] + [f'txn_type_{c}' for c in type_counts.columns[1:]]
    type_div = type_div.merge(type_counts, on='UniqueID', how='left')
    print(f'Type features: {type_div.shape}')
else:
    type_div = None
    print('No TransactionType column')

Trend features built


In [ ]:
# ============================================================
# BLOCK 7: Financial features
# ============================================================
fin = pd.read_parquet('../data/raw/financials_features.parquet')
fin['RunDate'] = pd.to_datetime(fin['RunDate'])
max_fin_date = fin['RunDate'].max()

# All-time aggregates
fin_all = fin.groupby('UniqueID').agg(
    avg_income    = ('NetInterestIncome', 'mean'),
    total_income  = ('NetInterestIncome', 'sum'),
    std_income    = ('NetInterestIncome', 'std'),
    avg_revenue   = ('NetInterestRevenue', 'mean'),
    total_revenue = ('NetInterestRevenue', 'sum'),
    std_revenue   = ('NetInterestRevenue', 'std'),
).reset_index()

# Recent 3m and 6m financial snapshots
for months, label in [(3, '3m'), (6, '6m')]:
    cutoff = max_fin_date - pd.DateOffset(months=months)
    recent_fin = fin[fin['RunDate'] >= cutoff].groupby('UniqueID').agg(
        avg_income  = ('NetInterestIncome', 'mean'),
        avg_revenue = ('NetInterestRevenue', 'mean'),
    ).reset_index()
    recent_fin.columns = ['UniqueID', f'fin_avg_income_{label}', f'fin_avg_revenue_{label}']
    fin_all = fin_all.merge(recent_fin, on='UniqueID', how='left')

# Per-product income (pivot top products)
top_products = fin['Product'].value_counts().nlargest(4).index
prod_income = fin[fin['Product'].isin(top_products)].groupby(
    ['UniqueID', 'Product']
)['NetInterestIncome'].mean().unstack(fill_value=0).reset_index()
prod_income.columns = ['UniqueID'] + [f'prod_income_{c}' for c in prod_income.columns[1:]]

# Number of products
product_count = fin.groupby('UniqueID')['Product'].nunique().reset_index(name='num_products')

fin_features = fin_all.merge(prod_income, on='UniqueID', how='left').merge(product_count, on='UniqueID', how='left')
print(f'Financial features: {fin_features.shape}')

Recency/tenure features built


In [ ]:
# ============================================================
# BLOCK 8: Demographics
# ============================================================
demo = pd.read_parquet('../data/raw/demographics_clean.parquet')
demo = demo.replace(['None', 'Not Disclosed / Unknown', 'Other / Unclassified'], np.nan)

demo['BirthDate'] = pd.to_datetime(demo['BirthDate'], errors='coerce')
ref_date = pd.to_datetime('2015-10-01')
demo['age'] = (ref_date - demo['BirthDate']).dt.days // 365
demo.loc[(demo['age'] < 18) | (demo['age'] > 100), 'age'] = np.nan
demo['age'] = demo['age'].fillna(demo['age'].median())

# Age buckets — often more useful than raw age for tree models
demo['age_bucket'] = pd.cut(demo['age'], bins=[0,25,35,45,55,65,100], labels=['18-25','26-35','36-45','46-55','56-65','65+'])

top_cities = demo['ResidentialCityName'].value_counts().nlargest(10).index
demo['ResidentialCityName'] = np.where(demo['ResidentialCityName'].isin(top_cities), demo['ResidentialCityName'], 'Other')

demo_selected = demo[['UniqueID', 'age', 'age_bucket', 'Gender', 'IncomeCategory',
                       'OccupationCategory', 'IndustryCategory',
                       'ResidentialCityName', 'CustomerBankingType']]

demo_ids = demo_selected[['UniqueID']]
demo_enc  = pd.get_dummies(demo_selected.drop(columns=['UniqueID']), drop_first=True)
demo_encoded = pd.concat([demo_ids, demo_enc], axis=1)
demo_encoded.columns = demo_encoded.columns.str.replace('[^A-Za-z0-9_]+', '_', regex=True)

print(f'Demo features: {demo_encoded.shape}')

No TransactionType column — skipping
Diversity/consistency features built


In [ ]:
# ============================================================
# Merge everything
# ============================================================
def build_dataset(base_df):
    df = base_df.copy()

    for feat in [feat_all, feat_debit, feat_credit,
                 feat_12m, feat_6m, feat_3m, feat_1m,
                 monthly_pivot, consistency,
                 feat_holiday, trend_df, trend_12m,
                 last_txn, first_txn, avg_per_month,
                 fin_features, demo_encoded]:
        df = df.merge(feat, on='UniqueID', how='left')

    if type_div is not None:
        df = df.merge(type_div, on='UniqueID', how='left')

    # Derived ratios
    df['debit_credit_ratio']    = (df['debit_count'].fillna(0) + 1) / (df['credit_count'].fillna(0) + 1)
    df['recent_activity_ratio'] = (df['txn_count_3m'].fillna(0) + 1) / (df['txn_count_6m'].fillna(0) + 1)
    df['activity_6m_12m_ratio'] = (df['txn_count_6m'].fillna(0) + 1) / (df['txn_count_12m'].fillna(0) + 1)
    df['holiday_share']         = df['holiday_txn_count'].fillna(0) / (df['txn_count_all'].fillna(0) + 1)

    df = df.fillna(0)
    return df

train_merged = build_dataset(train)
test_merged  = build_dataset(test)

print(f'Train: {train_merged.shape}')
print(f'Test:  {test_merged.shape}')

Financials: 372,245 rows | Date range: 2013-12-25 00:00:00 to 2015-10-31 00:00:00
['UniqueID', 'AccountID', 'RunDate', 'Product', 'NetInterestIncome', 'NetInterestRevenue']


In [ ]:
# ============================================================
# Merge everything
# ============================================================
def build_dataset(base_df):
    df = base_df.copy()

    for feat in [feat_all, feat_debit, feat_credit,
                 feat_12m, feat_6m, feat_3m, feat_1m,
                 monthly_pivot, consistency,
                 feat_holiday, trend_df, trend_12m,
                 last_txn, first_txn, avg_per_month,
                 fin_features, demo_encoded]:
        df = df.merge(feat, on='UniqueID', how='left')

    if type_div is not None:
        df = df.merge(type_div, on='UniqueID', how='left')

    # Derived ratios
    df['debit_credit_ratio']    = (df['debit_count'].fillna(0) + 1) / (df['credit_count'].fillna(0) + 1)
    df['recent_activity_ratio'] = (df['txn_count_3m'].fillna(0) + 1) / (df['txn_count_6m'].fillna(0) + 1)
    df['activity_6m_12m_ratio'] = (df['txn_count_6m'].fillna(0) + 1) / (df['txn_count_12m'].fillna(0) + 1)
    df['holiday_share']         = df['holiday_txn_count'].fillna(0) / (df['txn_count_all'].fillna(0) + 1)

    df = df.fillna(0)
    return df

train_merged = build_dataset(train)
test_merged  = build_dataset(test)

print(f'Train: {train_merged.shape}')
print(f'Test:  {test_merged.shape}')

Train features: (8360, 92)
Test features:  (3584, 91)


In [ ]:
# ============================================================
# Model: LightGBM with OOF + early stopping
# ============================================================
from lightgbm import LGBMRegressor, early_stopping, log_evaluation
from sklearn.model_selection import KFold

FEATURE_COLS = [c for c in train_merged.columns if c not in ['UniqueID', 'next_3m_txn_count']]
X      = train_merged[FEATURE_COLS]
y      = train_merged['next_3m_txn_count']
y_log  = np.log1p(y)
X_test = test_merged[FEATURE_COLS]

print(f'Features: {len(FEATURE_COLS)} | Train rows: {len(X)}')

lgbm_params = dict(
    n_estimators     = 3000,
    learning_rate    = 0.01,    # slower = better generalisation
    num_leaves       = 63,
    max_depth        = -1,
    min_child_samples= 20,
    subsample        = 0.8,
    subsample_freq   = 1,
    colsample_bytree = 0.7,
    reg_alpha        = 0.05,
    reg_lambda       = 0.1,
    random_state     = 42,
    n_jobs           = -1,
    verbose          = -1,
)

kf           = KFold(n_splits=5, shuffle=True, random_state=42)
oof_preds    = np.zeros(len(X))
test_preds   = np.zeros(len(X_test))
rmsle_scores = []

for fold, (tr_idx, val_idx) in enumerate(kf.split(X)):
    X_tr,  X_val  = X.iloc[tr_idx],     X.iloc[val_idx]
    y_tr,  y_val  = y_log.iloc[tr_idx], y_log.iloc[val_idx]

    model = LGBMRegressor(**lgbm_params)
    model.fit(
        X_tr, y_tr,
        eval_set=[(X_val, y_val)],
        callbacks=[early_stopping(100, verbose=False), log_evaluation(period=-1)]
    )

    val_pred          = model.predict(X_val)
    oof_preds[val_idx] = val_pred
    test_preds        += model.predict(X_test) / 5

    rmsle = np.sqrt(np.mean((val_pred - y_val.values) ** 2))
    rmsle_scores.append(rmsle)
    print(f'Fold {fold+1} | iters={model.best_iteration_} | RMSLE={rmsle:.4f}')

print(f'\nMean CV RMSLE: {np.mean(rmsle_scores):.4f} ± {np.std(rmsle_scores):.4f}')

Features: 90
Training rows: 8360


In [ ]:
# Feature importance
fi = pd.DataFrame({'feature': FEATURE_COLS, 'importance': model.feature_importances_})
fi = fi.sort_values('importance', ascending=False)
print(fi.head(25).to_string(index=False))

              feature  importance
         txn_count_6m        1103
         txn_count_1m         805
         txn_count_3m         787
    avg_txn_per_month         591
    holiday_txn_count         542
      txn_trend_ratio         500
          debit_count         409
 customer_tenure_days         378
recent_activity_ratio         368
        txn_count_all         360
   debit_credit_ratio         359
         credit_count         355
                  age         346
           debit_mean         331
       txn_median_all         327
  days_since_last_txn         311
       txn_trend_diff         308
           std_income         301
          credit_mean         278
            debit_sum         268


## 4. Create a Valid Submission

Your submission must match the format of SampleSubmission.csv exactly.

In [20]:
# Zindi wants log1p(raw_counts) — NOT the raw counts
test_preds_log = test_preds  # test_preds is already in log1p space from the model!
# No need for expm1 at all

submission = pd.read_csv('../data/raw/SampleSubmission.csv')
submission['next_3m_txn_count'] = test_preds_log  # values like 3.2, 4.1, 5.6

submission.to_csv('submission_v2.csv', index=False)
print(submission['next_3m_txn_count'].describe())

count    3584.000000
mean        4.389243
std         1.163811
min         0.994081
25%         3.594017
50%         4.618344
75%         5.241109
max         6.935441
Name: next_3m_txn_count, dtype: float64


## Local Scoring

You can score your submission locally using the included evaluate.py script:



Note: PublicReference.csv is only available if you have it for local testing. On Zindi, scoring is automatic.